In [3]:
import json
import os
import sys
import time
import requests
import replicate # pip install replicate

# ==================================================================================
# 1. CONFIGURATION
# ==================================================================================
JSON_FILE_PATH = "eeg_results_text.json"
OUTPUT_DIR = "output_videos_replicate"

# PASTE YOUR REPLICATE API TOKEN HERE
# or set it in your terminal: export REPLICATE_API_TOKEN=r8_...
api_token = "" 
os.environ["REPLICATE_API_TOKEN"] = api_token

# Model: Zeroscope v2 XL (High quality text-to-video)
MODEL_ID = "bytedance/seedance-1-pro-fast"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==================================================================================
# 2. MAIN LOOP
# ==================================================================================
if __name__ == "__main__":
    if "YOUR_REPLICATE_TOKEN" in os.environ.get("REPLICATE_API_TOKEN", ""):
        print("Error: You must set your REPLICATE_API_TOKEN in the script.")
        print("Get it here: https://replicate.com/account/api-tokens")
        sys.exit(1)

    if not os.path.exists(JSON_FILE_PATH):
        print(f"Error: Could not find {JSON_FILE_PATH}.")
        sys.exit(1)
        
    print("Loading JSON data...")
    with open(JSON_FILE_PATH, 'r') as f:
        data = json.load(f)
    data_lookup = {item['index']: item for item in data}

    while True:
        user_input = input("\n[Replicate Video] Enter Index (or 'q' to quit): ")
        if user_input.lower() == 'q': break
        
        try:
            target_idx = int(user_input)
        except ValueError:
            continue

        entry = data_lookup.get(target_idx)
        if not entry:
            print("Index not found.")
            continue
            
        prompt = entry['predicted_text']
        gt_text = entry['ground_truth_text']
        
        print(f"\n--- Sample {target_idx} ---")
        print(f"Ground Truth: '{gt_text}'")
        print(f"Sending prompt to Replicate: '{prompt}'")
        
        try:
            # 1. Run the model
            print("Running Zeroscope XL on the cloud... (wait approx 10-20s)")
            output = replicate.run(
                MODEL_ID,
                input={
                    "prompt": prompt,
                    "num_frames": 24,
                    "width": 576,
                    "height": 320,
                    "fps": 10
                }
            )
            
            # Replicate returns a list of URLs (usually just one video URL)
            video_url = output[0] if isinstance(output, list) else output
            print(f"Video generated! Downloading from: {video_url}")

            # 2. Download the Video
            response = requests.get(video_url)
            
            if response.status_code == 200:
                safe_prompt = "".join([c if c.isalnum() else "_" for c in prompt])[:20]
                filename = f"idx_{target_idx}_{safe_prompt}.mp4"
                save_path = os.path.join(OUTPUT_DIR, filename)
                
                with open(save_path, "wb") as f:
                    f.write(response.content)
                print(f"SUCCESS! Video saved to: {save_path}")
            else:
                print("Failed to download video file.")

        except Exception as e:
            print(f"Error: {e}")
            print("Check your API token and billing status.")

Loading JSON data...



[Replicate Video] Enter Index (or 'q' to quit):  4



--- Sample 4 ---
Ground Truth: 'a city at night with buildings lit up'
Sending prompt to Replicate: 'a city with tall buildings'
Running Zeroscope XL on the cloud... (wait approx 10-20s)
Error: ReplicateError Details:
title: Insufficient credit
status: 402
detail: You have insufficient credit to run this model. Go to https://replicate.com/account/billing#billing to purchase credit. Once you purchase credit, please wait a few minutes before trying again.
Check your API token and billing status.


KeyboardInterrupt: Interrupted by user